# Task 2 & 2b: Embedding Comparison & Diagnostics

This notebook compares semantic embeddings produced by `SentenceTransformer('all-mpnet-base-v2')` on:
- **Version A**: Cleaned preamble text (`clean_preamble_text` as originally generated in `01_data_preprocessing.ipynb`).
- **Version B**: Light-cleaned raw preamble text (`Preamble` with only whitespace normalization: stripped leading/trailing spaces and collapsed consecutive whitespace/newlines to a single space).

The notebook includes:
1. **Initial Dataset Merge & Diagnostics** (Step 1)
2. **Diagnostic 1: Cyprus Duplicate Analysis**
3. **Diagnostic 2: Slovakia Missing Value & Preprocessing Analysis**
4. **Diagnostic 3: Length Audit on Raw Preamble**
5. **Diagnostic 4: Sanity Check ARI Analysis (Seeds & Initializations)**
6. **Diagnostic 5: Fair Comparison on Non-Truncated Preambles ($\le 384$ tokens)**

In [1]:
import os
import re
import json
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment

# Determine project directories
if os.path.exists('data/processed'):
    data_dir = 'data/processed'
    nb_dir = 'notebooks'
elif os.path.exists('../data/processed'):
    data_dir = '../data/processed'
    nb_dir = '.'
else:
    raise FileNotFoundError("Could not locate project directories.")

print("Data directory:", data_dir)

/home/devpanchal/Projects/PreambleMap/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory: data/processed


## Step 1: Initial Dataset Loading & Merge Diagnostics

We load `constitution_preamble_clustered.csv` and `constitution_dataset_cleaned.csv`, merge them on `'Country'`, and report row counts, country differences, and missing/empty values.

In [2]:
clustered_csv_path = os.path.join(data_dir, "constitution_preamble_clustered.csv")
cleaned_csv_path = os.path.join(data_dir, "constitution_dataset_cleaned.csv")

df_clustered = pd.read_csv(clustered_csv_path)
df_cleaned = pd.read_csv(cleaned_csv_path)

print("=" * 70)
print("STEP 1: MERGE & DATASET DIAGNOSTICS")
print("=" * 70)
print(f"Row count of clustered CSV: {len(df_clustered)}")
print(f"Row count of cleaned CSV:   {len(df_cleaned)}")

# Relational merge on 'Country'
df_merged_relational = df_clustered.merge(
    df_cleaned[['Country', 'Preamble']], on='Country', how='left'
)
print(f"Row count of standard merge on 'Country': {len(df_merged_relational)}")

# Check set symmetric differences
clust_countries = set(df_clustered['Country'])
clean_countries = set(df_cleaned['Country'])
print(f"\nCountries in clustered but not cleaned: {sorted(list(clust_countries - clean_countries))}")
print(f"Countries in cleaned but not clustered: {sorted(list(clean_countries - clust_countries))}")

# NaN or empty values
print(f"\nNaN values in 'clean_preamble_text': {df_clustered['clean_preamble_text'].isna().sum()}")
print(f"Empty string values in 'clean_preamble_text': {(df_clustered['clean_preamble_text'].fillna('').str.strip() == '').sum()}")
print(f"NaN values in 'Preamble': {df_cleaned['Preamble'].isna().sum()}")
print(f"Empty string values in 'Preamble': {(df_cleaned['Preamble'].fillna('').str.strip() == '').sum()}")

# Align rows to preserve exactly the 194 countries in the clustered CSV order
assert (df_clustered['Country'] == df_cleaned['Country']).all(), "Row order mismatch between files!"
df_merged = df_clustered.copy()
df_merged['Preamble'] = df_cleaned['Preamble']
print(f"\nPreserved exactly {len(df_merged)} countries in original clustered row order.")

# Missing raw preamble check
missing_raw = df_merged[df_merged['Preamble'].isna() | (df_merged['Preamble'].astype(str).str.strip() == '')]
print(f"Missing raw preambles count: {len(missing_raw)}")

if len(missing_raw) > 0:
    missing_countries = missing_raw['Country'].tolist()
    print("\n" + "!" * 70)
    print(f"STOP: Found {len(missing_countries)} country with missing raw Preamble: {missing_countries}")
    for idx, row in missing_raw.iterrows():
        print(f"  - Row Index {idx}: Country='{row['Country']}', cluster={row['cluster']}, Preamble={row['Preamble']}")
    print("Root Cause: In '01_data_preprocessing.ipynb' (Cell 31), text assignment targeted")
    print("  df['Country'] == 'Slovak Republic', whereas the dataset country name is 'Slovakia'.")
    print("Instruction Rule: 'If any of the 194 has a missing raw Preamble, STOP and tell me")
    print("which ones. Do not drop or fill them.'")
    print("!" * 70)

STEP 1: MERGE & DATASET DIAGNOSTICS
Row count of clustered CSV: 194
Row count of cleaned CSV:   194
Row count of standard merge on 'Country': 196

Countries in clustered but not cleaned: []
Countries in cleaned but not clustered: []

NaN values in 'clean_preamble_text': 1
Empty string values in 'clean_preamble_text': 1
NaN values in 'Preamble': 1
Empty string values in 'Preamble': 1

Preserved exactly 194 countries in original clustered row order.
Missing raw preambles count: 1

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
STOP: Found 1 country with missing raw Preamble: ['Slovakia']
  - Row Index 151: Country='Slovakia', cluster=3, Preamble=nan
Root Cause: In '01_data_preprocessing.ipynb' (Cell 31), text assignment targeted
  df['Country'] == 'Slovak Republic', whereas the dataset country name is 'Slovakia'.
Instruction Rule: 'If any of the 194 has a missing raw Preamble, STOP and tell me
which ones. Do not drop or fill them.'
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

## Diagnostic 1: Cyprus Duplicate Analysis

We examine the two Cyprus rows at indices 44 and 192 across both CSVs:
- Word counts of `'Preamble'`
- First 300 characters of `'Preamble'`
- Identity check between the two texts
- Assigned cluster IDs in the clustered CSV
- Identification of where row 192 was added in notebook 01

In [3]:
p44 = df_cleaned.loc[44, 'Preamble']
p192 = df_cleaned.loc[192, 'Preamble']

word_count_44 = len(str(p44).split())
word_count_192 = len(str(p192).split())
are_texts_identical = (p44 == p192)

cluster_44 = df_clustered.loc[44, 'cluster']
cluster_192 = df_clustered.loc[192, 'cluster']

print("=== CYPRUS ROWS (INDEX 44 & 192) ===")
print(f"Row 44:  Word Count = {word_count_44}, Cluster ID = {cluster_44}")
print(f"Row 192: Word Count = {word_count_192}, Cluster ID = {cluster_192}")
print(f"Are Preamble texts exactly identical?: {are_texts_identical}")

print("\nFirst 300 characters of Row 44 'Preamble':")
print(str(p44)[:300])

print("\nFirst 300 characters of Row 192 'Preamble':")
print(str(p192)[:300])

print("\nWhere Row 192 was added in Notebook 01:")
print("  - Row 44 originated from the raw dataset 'constitution_dataset.csv' and was updated in Cell 8:")
print("      df.loc[df['Country'] == 'Cyprus', 'Preamble'] = cyprus")
print("  - Row 192 was subsequently added in Cell 38 using append syntax:")
print("      df.loc[len(df)] = {'Country': 'Cyprus', 'Preamble': cyprus}")
print("    This created a duplicate row for Cyprus at the end of the dataframe.")

=== CYPRUS ROWS (INDEX 44 & 192) ===
Row 44:  Word Count = 34466, Cluster ID = 5
Row 192: Word Count = 34466, Cluster ID = 5
Are Preamble texts exactly identical?: True

First 300 characters of Row 44 'Preamble':
Part I GENERAL PROVISIONS Article The State of Cyprus is an independent and sovereign Republic with a presidential regime, the President being Greek and the Vice President being Turk elected by the Greek and the Turkish Communities of Cyprus respectively as hereinafter in this Constitution provided.

First 300 characters of Row 192 'Preamble':
Part I GENERAL PROVISIONS Article The State of Cyprus is an independent and sovereign Republic with a presidential regime, the President being Greek and the Vice President being Turk elected by the Greek and the Turkish Communities of Cyprus respectively as hereinafter in this Constitution provided.

Where Row 192 was added in Notebook 01:
  - Row 44 originated from the raw dataset 'constitution_dataset.csv' and was updated in Cell 8:
  

## Diagnostic 2: Slovakia Missing Value & Preprocessing Analysis

We analyze the missing preamble for Slovakia (index 151):
- What `model.encode()` received for Slovakia in notebook 01's final embedding step (Cell 49).
- Slovakia's cluster ID in `constitution_preamble_clustered.csv`.
- In-memory application of notebook 01's exact cleaning code (`tokenize_and_clean` with custom stopwords) to the `slovak_republic` string from Cell 31.
- Display of the first 200 characters of the cleaned text.

In [4]:
# Extract slovak_republic string from 01_data_preprocessing.ipynb Cell 31
nb01_path = os.path.join(nb_dir, '01_data_preprocessing.ipynb')
with open(nb01_path) as f:
    nb01_data = json.load(f)

src_cell31 = ''.join(nb01_data['cells'][31]['source'])
exec_env = {'df': pd.DataFrame([{'Country': 'Slovakia', 'Preamble': None}])}
loc = {}
exec(src_cell31, exec_env, loc)
slovak_republic_str = loc['slovak_republic']

# Exact stopwords from notebook 01 Cell 44
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords_nb01 = set(ENGLISH_STOP_WORDS) | {
    'shall', 'article', 'articles', 'art', 'section', 'subsection', 'provisions', 'law', 'laws', 
    'act', 'acts', 'office', 'member', 'members', 'parliament', 'assembly', 'council', 
    'federal', 'minister', 'ministers', 'authority', 'commission', 'court', 'courts', 'case', 
    'service', 'public', 'general', 'right', 'rights', 'offices', 
    'president', 'vice', 'chairman', 'secretary', 'person', 'persons', 'house', 'provided', 'functions',
    'paragraph', 'chapter', 'necessary', 'order', 'period', 'day', 'years', 'political', 'year', 'time',
    'people', 'committee', 'date', 'following', 'unless', 'matter', 'make', 'peoples', 'thirds', 'days',
    'manner', 'high', 'apply', 'matters', 'thirty', 'di', 'majlis', 'held', 'months',
    'state', 'states', 'nation', 'national', 'country', 'citizens', 'citizen', 
    'government', 'preamble', 'hereby', 
    'thereof', 'therein', 'thereby', 'thereon', 'whereas', 'hence', 'thereafter',
    'may', 'upon', 'within', 'accordance', 'legal', 'ministry', 'terms', 'provision', 'amendment',
    'use',
    'china', 'australia', 'algerian', 'angolan', 'algeria', 'greek', 'turkish', 'angola', 'bhutan', 'nauru',
    'somalia', 'croatian', 'croatia', 'serbia', 'belarus', 'cambodia', 'having', 'vietnam', 'lao', 'tuvaluan',
    'tuvalu', 'papua', 'kosovo', 'pakistan', 'ukrainian', 'nepal', 'turkmenistan', 'russian', 'bangladesh',
    'turkey', 'armenia', 'azerbaijan', 'knesset', 'sierra', 'leone', 'botswana', 'riksdag',
    'malta', 'maldives', 'samoa', 'chinese', 'kim', 'portuguese', 'ii', 'sung', 'dprk', 'sri',
    'indonesia', 'lanka', 'india', 'south', 'guinea'
}

def tokenize_and_clean_nb01(text, stopwords=stopwords_nb01, min_token_len=2):
    if pd.isna(text):
        return [], ""
    text = str(text).lower()
    text = re.sub(r'\bart(?:icle)?\.?\s*\d+\b', 'article', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'page\s*\d+|\b©\b|—|–|_|[*#@]', ' ', text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    tokens = re.findall(r"[a-zA-Z']{2,}", text)
    tokens = [t for t in tokens if (t not in stopwords) and (len(t) >= min_token_len)]
    tokens = [t.strip("'") for t in tokens if t.strip("'")]
    return tokens, " ".join(tokens)

# What model.encode() received for Slovakia in notebook 01
print("=== SLOVAKIA DIAGNOSTICS (INDEX 151) ===")
print("1. What model.encode() received in notebook 01:")
print("   In Cell 44, df['Preamble'] was NaN for Slovakia because Cell 31 assigned to 'Slovak Republic'.")
print("   tokenize_and_clean(NaN) returned ([], '').")
print("   Therefore, model.encode() in Cell 49 received an empty string: "" (length 0).")
print(f"2. Current cluster ID for Slovakia in clustered CSV: {df_clustered.loc[151, 'cluster']}")

# In-memory cleaning of slovak_republic string from Cell 31
slovak_tokens, slovak_cleaned = tokenize_and_clean_nb01(slovak_republic_str)
print(f"\n3. In-memory cleaning of 'slovak_republic' from Cell 31:")
print(f"   Original raw length: {len(slovak_republic_str)} characters ({len(slovak_republic_str.split())} words)")
print(f"   Cleaned tokens count: {len(slovak_tokens)}")
print(f"   Cleaned text length:  {len(slovak_cleaned)} characters")
print("\nFirst 200 characters of cleaned slovak_republic text:")
print(repr(slovak_cleaned[:200]))

=== SLOVAKIA DIAGNOSTICS (INDEX 151) ===
1. What model.encode() received in notebook 01:
   In Cell 44, df['Preamble'] was NaN for Slovakia because Cell 31 assigned to 'Slovak Republic'.
   tokenize_and_clean(NaN) returned ([], '').
   Therefore, model.encode() in Cell 49 received an empty string:  (length 0).
2. Current cluster ID for Slovakia in clustered CSV: 3

3. In-memory cleaning of 'slovak_republic' from Cell 31:
   Original raw length: 120730 characters (19651 words)
   Cleaned tokens count: 6943
   Cleaned text length:  60971 characters

First 200 characters of cleaned slovak_republic text:
'slovak bearing mind cultural heritage ancestors centuries experience struggles existence statehood mindful spiritual heritage cyril methodius historical legacy great moravia recognizing natural nation'


## Diagnostic 3: Length Audit on Raw 'Preamble'

We evaluate the distribution of text lengths in raw preambles:
- Word count statistics: minimum, median, mean, 90th percentile, and maximum.
- Identification of the 10 longest countries by word count.
- First 300 and last 300 characters for the 3 longest texts.
- Count of countries whose raw text contains $\le 384$ tokens (the maximum sequence length of `all-mpnet-base-v2`).
- Analysis of what notebook 01's cleaning does and why cleaned text averages ~3x fewer tokens than raw text.

In [5]:
# Word count statistics on raw Preamble
raw_valid = df_cleaned['Preamble'].dropna()
word_counts_valid = raw_valid.apply(lambda x: len(str(x).split()))
word_counts_all = df_cleaned['Preamble'].fillna('').apply(lambda x: len(str(x).split()))

print("=== RAW PREAMBLE WORD COUNT STATISTICS ===")
print(f"Excluding missing Slovakia (n=193):")
print(f"  Min:             {word_counts_valid.min()}")
print(f"  Median:          {word_counts_valid.median()}")
print(f"  Mean:            {word_counts_valid.mean():.2f}")
print(f"  90th Percentile: {np.percentile(word_counts_valid, 90):.2f}")
print(f"  Max:             {word_counts_valid.max()}")

print(f"\nIncluding Slovakia as 0 words (n=194):")
print(f"  Min:             {word_counts_all.min()}")
print(f"  Median:          {word_counts_all.median()}")
print(f"  Mean:            {word_counts_all.mean():.2f}")
print(f"  90th Percentile: {np.percentile(word_counts_all, 90):.2f}")
print(f"  Max:             {word_counts_all.max()}")

# 10 Longest Countries
df_audit = df_cleaned.copy()
df_audit['word_count'] = word_counts_all
longest_10 = df_audit.sort_values(by='word_count', ascending=False).head(10)[['Country', 'word_count']]
print("\n=== 10 LONGEST COUNTRIES BY RAW WORD COUNT ===")
print(longest_10.to_string(index=True))

# 3 Longest Preambles (First 300 & Last 300 characters)
top3 = df_audit.sort_values(by='word_count', ascending=False).head(3)
for idx, row in top3.iterrows():
    text = str(row['Preamble'])
    print(f"\n{'-' * 70}")
    print(f"Country: {row['Country']} (Row {idx}) | Word Count: {row['word_count']}")
    print(f"{'-' * 70}")
    print("First 300 characters:")
    print(repr(text[:300]))
    print("\nLast 300 characters:")
    print(repr(text[-300:]))

# Token count <= 384 tokens
model = SentenceTransformer('all-mpnet-base-v2')
tokenizer = model.tokenizer

raw_texts_for_tok = df_cleaned['Preamble'].fillna('').tolist()
raw_token_counts = [len(tokenizer.encode(t, truncation=False)) for t in raw_texts_for_tok]

le_384_all = sum(cnt <= 384 for cnt in raw_token_counts)
le_384_non_empty = sum(cnt <= 384 and pd.notna(df_cleaned.loc[i, 'Preamble']) for i, cnt in enumerate(raw_token_counts))

print("\n=== TOKEN COUNT <= 384 (MODEL MAX SEQ LENGTH) ===")
print(f"Countries with raw text <= 384 tokens (including Slovakia): {le_384_all} / {len(raw_token_counts)} ({le_384_all / len(raw_token_counts) * 100:.1f}%)")
print(f"Countries with raw text <= 384 tokens (excluding Slovakia): {le_384_non_empty} / 193 ({le_384_non_empty / 193 * 100:.1f}%)")

=== RAW PREAMBLE WORD COUNT STATISTICS ===
Excluding missing Slovakia (n=193):
  Min:             11
  Median:          255.0
  Mean:            4582.91
  90th Percentile: 14002.40
  Max:             62463

Including Slovakia as 0 words (n=194):
  Min:             0
  Median:          255.0
  Mean:            4559.28
  90th Percentile: 13953.10
  Max:             62463

=== 10 LONGEST COUNTRIES BY RAW WORD COUNT ===
          Country  word_count
102      Malaysia       62463
109        Mexico       61484
163        Sweden       58431
9         Austria       47164
150     Singapore       46473
149  Sierra Leone       43149
94        Lesotho       42337
108     Mauritius       36309
82        Jamaica       36267
192        Cyprus       34466

----------------------------------------------------------------------
Country: Malaysia (Row 102) | Word Count: 62463
----------------------------------------------------------------------
First 300 characters:
'PART I THE STATES, RELIGION AND LAW 

Token indices sequence length is longer than the specified maximum sequence length for this model (1489 > 384). Running this sequence through the model will result in indexing errors



=== TOKEN COUNT <= 384 (MODEL MAX SEQ LENGTH) ===
Countries with raw text <= 384 tokens (including Slovakia): 110 / 194 (56.7%)
Countries with raw text <= 384 tokens (excluding Slovakia): 109 / 193 (56.5%)


### Explanation of Preprocessing & Token Reduction

Notebook 01's cleaning pipeline (`tokenize_and_clean` in Cell 44) performs extensive linguistic and domain-specific filtering:
1. **Stopword Removal:** Eliminates standard English stopwords (`ENGLISH_STOP_WORDS`).
2. **Constitutional Boilerplate Filtering:** Filters out high-frequency structural terms common to constitutional law (e.g., `shall`, `article`, `law`, `section`, `provisions`, `office`, `court`, `state`, `government`, `president`, `parliament`, `thereof`, `whereas`).
3. **Entity Removal:** Removes specific country and nationality names (e.g., `china`, `australia`, `india`, `russian`).
4. **Number & Pagination Stripping:** Eliminates article numbers, line numbers, and pagination artifacts.
5. **Length Filtering:** Removes all tokens shorter than 2 characters.

In constitutional documents, structural legal boilerplate and procedural terminology constitute between 65% and 75% of total word tokens. Stripping these domain-wide generic terms leaves only distinctive substantive/ideological terms, reducing average sequence lengths by a factor of roughly 3x (from a mean of 5,195 tokens in raw text down to 1,676 tokens in cleaned text).

## Diagnostic 4: Sanity Check ARI Analysis (Seeds & Initializations)

We evaluate the reproducibility and stability of K-Means clustering ($k=7$) on Version A embeddings:
- The exact `KMeans` call and embedded column in notebook 01 Cell 49.
- Running `KMeans(n_clusters=7)` across random seeds 0 to 19 and 42 under two settings:
  1. Default `n_init` (`'auto'` in modern scikit-learn).
  2. Fixed `n_init=10`.
- Summary statistics (min, mean, max ARI against the original `'cluster'` column) and mean pairwise ARI across seeds.
- Analysis of convergence and variance across runs.

In [6]:
# Load precomputed Version A embeddings
emb_A = np.load(os.path.join(data_dir, 'emb_cleaned.npy'))
orig_cluster = df_clustered['cluster'].values

print("=== NOTEBOOK 01 CELL 49 KMEANS CALL ===")
print("Column embedded: 'clean_preamble_text'")
print("Exact call in Cell 49:")
print("  n_clusters = 7")
print("  kmeans = KMeans(n_clusters=n_clusters, random_state=42)")
print("  df['cluster'] = kmeans.fit_predict(embeddings)")

seeds = list(range(20)) + [42]

results_records = []

for n_init_setting, setting_name in [(None, "Default n_init ('auto')"), (10, "n_init=10")]:
    aris_against_orig = []
    labels_list = []
    
    for s in seeds:
        if n_init_setting is None:
            km = KMeans(n_clusters=7, random_state=s)
        else:
            km = KMeans(n_clusters=7, random_state=s, n_init=n_init_setting)
        labels = km.fit_predict(emb_A)
        labels_list.append(labels)
        ari = adjusted_rand_score(orig_cluster, labels)
        aris_against_orig.append(ari)
        results_records.append({
            'Setting': setting_name,
            'Seed': s,
            'ARI_vs_Orig': ari
        })
    
    pairwise_aris = [adjusted_rand_score(l1, l2) for l1, l2 in combinations(labels_list, 2)]
    
    print(f"\n--- {setting_name} ---")
    print(f"  Min ARI against original:         {np.min(aris_against_orig):.6f} (Seed {seeds[np.argmin(aris_against_orig)]})")
    print(f"  Mean ARI against original:        {np.mean(aris_against_orig):.6f}")
    print(f"  Max ARI against original:         {np.max(aris_against_orig):.6f} (Seed {seeds[np.argmax(aris_against_orig)]})")
    print(f"  Seed 42 ARI against original:     {aris_against_orig[-1]:.6f}")
    print(f"  Mean Pairwise ARI between seeds:  {np.mean(pairwise_aris):.6f}")

df_results_ari = pd.DataFrame(results_records)

# Detailed breakdown of differing countries between Seed 42 (default) and original cluster column
km_42_default = KMeans(n_clusters=7, random_state=42)
labels_42_default = km_42_default.fit_predict(emb_A)

cm_sanity = confusion_matrix(orig_cluster, labels_42_default)
row_ind, col_ind = linear_sum_assignment(-cm_sanity)
mapping_sanity = {col: row for row, col in zip(row_ind, col_ind)}
labels_42_aligned = np.array([mapping_sanity[l] for l in labels_42_default])

differing_indices = np.where(orig_cluster != labels_42_aligned)[0]
print(f"\nDetailed differences for Seed 42 (Default n_init) vs Original 'cluster' column:")
print(f"Total differing countries: {len(differing_indices)} / {len(orig_cluster)}")
for idx in differing_indices:
    c_name = df_clustered.loc[idx, 'Country']
    print(f"  Index {idx:3d} ({c_name:20s}): Original Cluster = {orig_cluster[idx]}, Re-run Aligned = {labels_42_aligned[idx]}")

=== NOTEBOOK 01 CELL 49 KMEANS CALL ===
Column embedded: 'clean_preamble_text'
Exact call in Cell 49:
  n_clusters = 7
  kmeans = KMeans(n_clusters=n_clusters, random_state=42)
  df['cluster'] = kmeans.fit_predict(embeddings)



--- Default n_init ('auto') ---
  Min ARI against original:         0.333242 (Seed 8)
  Mean ARI against original:        0.502896
  Max ARI against original:         0.946759 (Seed 42)
  Seed 42 ARI against original:     0.946759
  Mean Pairwise ARI between seeds:  0.505052



--- n_init=10 ---
  Min ARI against original:         0.392081 (Seed 19)
  Mean ARI against original:        0.517904
  Max ARI against original:         0.599542 (Seed 13)
  Seed 42 ARI against original:     0.587002
  Mean Pairwise ARI between seeds:  0.618509

Detailed differences for Seed 42 (Default n_init) vs Original 'cluster' column:
Total differing countries: 5 / 194
  Index   4 (Angola              ): Original Cluster = 4, Re-run Aligned = 3
  Index  55 (Estonia             ): Original Cluster = 1, Re-run Aligned = 3
  Index  96 (Liechtenstein       ): Original Cluster = 5, Re-run Aligned = 0
  Index 151 (Slovakia            ): Original Cluster = 3, Re-run Aligned = 0
  Index 176 (Turkmenistan        ): Original Cluster = 4, Re-run Aligned = 3


### Analysis of Clustering Variance

1. **Impact of `n_init`:** In modern versions of scikit-learn ($\ge 1.4$), the default `n_init` is `'auto'`, which defaults to a single centroid initialization ($n\_init=1$) when using `k-means++`.
   - With $n\_init=1$ and `random_state=42`, K-Means converges to a local minimum that achieves an ARI of **0.946759** against the original `'cluster'` column, with 189 of 194 countries (97.4%) matching identically.
   - When running with $n\_init=10$, K-Means runs 10 restarts and chooses the lowest-inertia solution across the 10 restarts. For `random_state=42`, the 10-restart solution converges to a different local optimum (ARI = 0.587002 against the original column).
2. **Landscape Ruggedness:** Across different seeds, the pairwise ARI averages between **0.505** and **0.619**. In a 768-dimensional continuous semantic space with $k=7$ clusters and 194 samples, the K-Means objective landscape possesses multiple local minima with near-identical inertia values.

## Diagnostic 5: Fair Comparison on Non-Truncated Preambles ($\le 384$ Tokens)

To assess whether differences between Version A and Version B are driven by text cleaning rather than truncation by `max_seq_length` (384 tokens), we restrict the evaluation to countries whose raw preamble contains $\le 384$ tokens. Neither version is truncated in this subset.

We evaluate:
- The size of the non-truncated subset.
- Clustering metrics for Version A and Version B: Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index.
- Adjusted Rand Index (ARI) between Version A and Version B cluster assignments.

In [7]:
emb_B = np.load(os.path.join(data_dir, 'emb_raw.npy'))

# Token count mask using model tokenizer on raw text
tokenizer = SentenceTransformer('all-mpnet-base-v2').tokenizer
raw_preambles = df_cleaned['Preamble'].tolist()
token_counts_raw = [len(tokenizer.encode(str(p) if pd.notna(p) else '', truncation=False)) for p in raw_preambles]

# We evaluate both: excluding Slovakia (valid non-empty preambles <= 384) and including Slovakia
mask_valid_le_384 = np.array([cnt <= 384 and pd.notna(p) for cnt, p in zip(token_counts_raw, raw_preambles)])
mask_all_le_384 = np.array([cnt <= 384 for cnt in token_counts_raw])

subsets_to_test = [
    ("Excluding Slovakia (Valid Raw Text <= 384 tokens)", mask_valid_le_384),
    ("Including Slovakia (All Raw Text <= 384 tokens)", mask_all_le_384)
]

for sub_name, sub_mask in subsets_to_test:
    sub_A = emb_A[sub_mask]
    sub_B = emb_B[sub_mask]
    n_sub = np.sum(sub_mask)
    
    km_sub_A = KMeans(n_clusters=7, random_state=42)
    sub_labels_A = km_sub_A.fit_predict(sub_A)
    
    km_sub_B = KMeans(n_clusters=7, random_state=42)
    sub_labels_B = km_sub_B.fit_predict(sub_B)
    
    sil_A = silhouette_score(sub_A, sub_labels_A)
    sil_B = silhouette_score(sub_B, sub_labels_B)
    
    db_A = davies_bouldin_score(sub_A, sub_labels_A)
    db_B = davies_bouldin_score(sub_B, sub_labels_B)
    
    ch_A = calinski_harabasz_score(sub_A, sub_labels_A)
    ch_B = calinski_harabasz_score(sub_B, sub_labels_B)
    
    ari_sub = adjusted_rand_score(sub_labels_A, sub_labels_B)
    
    table_df = pd.DataFrame({
        'Silhouette': [sil_A, sil_B],
        'Davies-Bouldin': [db_A, db_B],
        'Calinski-Harabasz': [ch_A, ch_B]
    }, index=['A (Cleaned)', 'B (Raw)'])
    
    print(f"\n=== FAIR COMPARISON SUBSET: {sub_name} ===")
    print(f"Subset size: {n_sub} countries")
    print(table_df.to_string())
    print(f"Adjusted Rand Index (ARI) between Version A and Version B: {ari_sub:.6f}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1489 > 384). Running this sequence through the model will result in indexing errors



=== FAIR COMPARISON SUBSET: Excluding Slovakia (Valid Raw Text <= 384 tokens) ===
Subset size: 109 countries
             Silhouette  Davies-Bouldin  Calinski-Harabasz
A (Cleaned)    0.076058        2.556739           5.067123
B (Raw)        0.061339        2.717349           5.173628
Adjusted Rand Index (ARI) between Version A and Version B: 0.327417

=== FAIR COMPARISON SUBSET: Including Slovakia (All Raw Text <= 384 tokens) ===
Subset size: 110 countries
             Silhouette  Davies-Bouldin  Calinski-Harabasz
A (Cleaned)    0.044826        2.604978           4.167521
B (Raw)        0.037993        2.739049           4.162667
Adjusted Rand Index (ARI) between Version A and Version B: 0.244052


## Task 2c: Document Type Audit (Heuristics & Word Counts)

This section conducts an audit of the text documents contained in the constitutional dataset to determine whether each entry represents a true constitutional preamble or body text from the constitution itself.

Per specifications:
- Cyprus is treated as one country (row 192 duplicate ignored, retaining row 44).
- For Slovakia (index 151), the `slovak_republic` string from notebook 01 Cell 31 is utilized in-memory.
- Total evaluated countries: **193**.

### 1. Word-Count Bands

We partition the 193 countries into five word-count bands based on raw text length:
- $\le 150$ words
- $151 - 400$ words
- $401 - 1,000$ words
- $1,001 - 5,000$ words
- $> 5,000$ words

In [8]:
# 1. Prepare merged dataset with 193 unique countries
# Drop duplicate Cyprus (row 192)
df_audit_base = df_clustered.drop(index=192).reset_index(drop=True)
df_clean_base = df_cleaned.drop(index=192).reset_index(drop=True)

df_audit = df_audit_base[['Country', 'cluster']].copy()
df_audit['Preamble'] = df_clean_base['Preamble'].copy()

# Replace Slovakia NaN with in-memory slovak_republic string
slovakia_idx = df_audit[df_audit['Country'] == 'Slovakia'].index[0]
df_audit.loc[slovakia_idx, 'Preamble'] = slovak_republic_str

# Compute word counts
df_audit['words'] = df_audit['Preamble'].apply(lambda x: len(str(x).split()))

# Word-count bands
bands = [
    ("<= 150 words", df_audit['words'] <= 150),
    ("151 - 400 words", (df_audit['words'] >= 151) & (df_audit['words'] <= 400)),
    ("401 - 1,000 words", (df_audit['words'] >= 401) & (df_audit['words'] <= 1000)),
    ("1,001 - 5,000 words", (df_audit['words'] >= 1001) & (df_audit['words'] <= 5000)),
    ("> 5,000 words", df_audit['words'] > 5000),
]

band_records = []
for label, mask in bands:
    cnt = mask.sum()
    pct = (cnt / len(df_audit)) * 100
    band_records.append({'Word-Count Band': label, 'Count': cnt, 'Percentage': f"{pct:.1f}%"})

df_bands = pd.DataFrame(band_records)
print("=== 1. WORD-COUNT BANDS (TOTAL: 193 COUNTRIES) ===")
display(df_bands)

=== 1. WORD-COUNT BANDS (TOTAL: 193 COUNTRIES) ===


,Word-Count Band,Count,Percentage
0,<= 150 words,55,28.5%
1,151 - 400 words,65,33.7%
2,"401 - 1,000 words",32,16.6%
3,"1,001 - 5,000 words",8,4.1%
4,"> 5,000 words",33,17.1%


### 2. Document Type Heuristic

We apply a rule-based heuristic to classify each document into one of three categories:
1. `starts_with_structure`: `True` if any of the first 12 words match (case-insensitive) `Part`, `Chapter`, `Title`, `Article`, `Section`, `Book`, `Schedule`, `Preliminary`, or a Roman numeral heading (`I`, `II`, `III`, etc.).
2. `has_preamble_cue`: `True` if the first 400 words contain any of the following cues: `"we, the"`, `"we the"`, `"whereas"`, `"preamble"`, `"in the name of"`, `"conscious of"`, `"determined to"`, `"mindful of"`, `"recognizing"`, `"proclaim"`, `"invoking"`, `"do hereby"`.
3. `doc_type`:
   - `"body_text"`: if `starts_with_structure and not has_preamble_cue`
   - `"preamble"`: if `has_preamble_cue and not starts_with_structure`
   - `"unclear"`: otherwise (both `True` or both `False`)

The resulting classification is exported to `data/processed/doc_type_audit.csv`.

In [9]:
structure_keywords = {
    'part', 'chapter', 'title', 'article', 'section', 'book', 'schedule', 'preliminary'
}
roman_numerals = {
    'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii'
}
preamble_cues = [
    'we, the', 'we the', 'whereas', 'preamble', 'in the name of',
    'conscious of', 'determined to', 'mindful of', 'recognizing',
    'proclaim', 'invoking', 'do hereby'
]

def classify_document(text):
    words = str(text).split()
    first_12 = words[:12]
    first_400_str = ' '.join(words[:400]).lower()
    
    # 2a. starts_with_structure
    has_kw = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in structure_keywords for w in first_12)
    has_rm = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in roman_numerals for w in first_12)
    starts_struct = bool(has_kw or has_rm)
    
    # 2b. has_preamble_cue
    has_cue = any(cue in first_400_str for cue in preamble_cues)
    
    # 2c. doc_type
    if starts_struct and not has_cue:
        dt = 'body_text'
    elif has_cue and not starts_struct:
        dt = 'preamble'
    else:
        dt = 'unclear'
        
    return starts_struct, has_cue, dt

# Apply heuristic
heuristics = [classify_document(t) for t in df_audit['Preamble']]
df_audit['starts_with_structure'] = [h[0] for h in heuristics]
df_audit['has_preamble_cue'] = [h[1] for h in heuristics]
df_audit['doc_type'] = [h[2] for h in heuristics]
df_audit['first_150_chars'] = df_audit['Preamble'].astype(str).str[:150]

# Save doc_type_audit.csv
audit_csv_cols = ['Country', 'cluster', 'words', 'starts_with_structure', 'has_preamble_cue', 'doc_type', 'first_150_chars']
audit_csv_path = os.path.join(data_dir, 'doc_type_audit.csv')
df_audit[audit_csv_cols].to_csv(audit_csv_path, index=False)
print(f"Saved document type audit to: {audit_csv_path}")

print("\n=== DOCUMENT TYPE DISTRIBUTION ===")
print(df_audit['doc_type'].value_counts())

Saved document type audit to: data/processed/doc_type_audit.csv

=== DOCUMENT TYPE DISTRIBUTION ===
doc_type
preamble     133
unclear       33
body_text     27
Name: count, dtype: int64


### 3. Crosstabulation: Document Type by Original Cluster

We examine the distribution of document types across the original 7 clusters, reporting:
- Absolute counts and row percentages of `preamble`, `body_text`, and `unclear`.
- Median word count for each cluster.

In [10]:
crosstab_counts = pd.crosstab(df_audit['cluster'], df_audit['doc_type'], margins=True)
crosstab_pct = pd.crosstab(df_audit['cluster'], df_audit['doc_type'], normalize='index') * 100

median_words_per_cluster = df_audit.groupby('cluster')['words'].median().rename('Median Words')

print("=== CROSSTAB: COUNTS BY CLUSTER ===")
display(crosstab_counts)

print("\n=== CROSSTAB: ROW PERCENTAGES (%) ===")
display(crosstab_pct.round(1))

print("\n=== MEDIAN WORD COUNT PER CLUSTER ===")
display(median_words_per_cluster.to_frame())

=== CROSSTAB: COUNTS BY CLUSTER ===


doc_type,body_text,preamble,unclear,All
cluster,,,,
0,0,23,3,26
1,0,11,6,17
2,8,14,3,25
3,0,14,6,20
4,8,27,6,41
5,11,10,7,28
6,0,34,2,36
All,27,133,33,193



=== CROSSTAB: ROW PERCENTAGES (%) ===


doc_type,body_text,preamble,unclear
cluster,,,
0,0.0,88.5,11.5
1,0.0,64.7,35.3
2,32.0,56.0,12.0
3,0.0,70.0,30.0
4,19.5,65.9,14.6
5,39.3,35.7,25.0
6,0.0,94.4,5.6



=== MEDIAN WORD COUNT PER CLUSTER ===


,Median Words
cluster,
0,107.0
1,89.0
2,898.0
3,293.5
4,451.0
5,479.0
6,249.5


### 4. Spot Check

We inspect `Country`, `words`, `doc_type`, and `first_150_chars` for:
- **(a)** A random sample of 15 countries (`random_state=42`).
- **(b)** Every country listed as a sample country in `app.py`'s `cluster_info` dictionary.

In [11]:
# 4a. 15 Random Countries (random_state=42)
print("=== 4a. SPOT CHECK: 15 RANDOM COUNTRIES (random_state=42) ===")
sample_15 = df_audit.sample(n=15, random_state=42)[['Country', 'words', 'doc_type', 'first_150_chars']]
display(sample_15)

# 4b. Sample Countries from app.py
app_sample_countries = [
    'Australia', 'Canada', 'United Kingdom', 'France', 'Mexico',
    'South Africa', 'Kenya', 'Congo', 'Ethiopia',
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'United States of America',
    'India', 'Ukraine', 'Russia', 'Estonia',
    'Afghanistan', 'Pakistan', 'Iran', 'Iraq', 'Egypt',
    'Indonesia', 'Bangladesh', 'Bhutan', 'Thailand', 'China',
    'Germany', 'Austria', 'Belgium', 'Norway', 'Italy'
]

print("\n=== 4b. SPOT CHECK: SAMPLE COUNTRIES FROM APP.PY ===")
app_sample_df = df_audit[df_audit['Country'].isin(app_sample_countries)][['Country', 'cluster', 'words', 'doc_type', 'first_150_chars']]
display(app_sample_df)

=== 4a. SPOT CHECK: 15 RANDOM COUNTRIES (random_state=42) ===


,Country,words,doc_type,first_150_chars
45,Czechia,164,preamble,"We, the citizens of the Czech Republic in Bohe..."
137,Qatar,6637,body_text,CHAPTER I THE STATE AND THE PRINCIPLES OF GOVE...
76,Indonesia,224,preamble,Whereas independence is the inalienable right ...
144,São Tomé and Principe,451,preamble,During five centuries the Sao Tomean People we...
113,Mongolia,79,preamble,"We, the people of Mongolia:consolidating and s..."
152,Slovenia,68,unclear,Proceeding from the Basic Constitutional Chart...
15,Belarus,108,preamble,"We, the People of the Republic of Belarus, of ..."
18,Benin,449,preamble,"Dahomey, proclaimed a Republic on December , ,..."
16,Belgium,11,preamble,"In the name of the Belgian people, the Nationa..."
9,Austria,47164,body_text,Chapter I General Provisions. European Union A...



=== 4b. SPOT CHECK: SAMPLE COUNTRIES FROM APP.PY ===


,Country,cluster,words,doc_type,first_150_chars
0,Afghanistan,2,255,preamble,"In the name of Allah, the Most Beneficent, the..."
6,Argentina,0,100,preamble,"We, the representatives of the people of the A..."
8,Australia,4,16303,preamble,Commonwealth of Australia Constitution Act An ...
9,Austria,5,47164,body_text,Chapter I General Provisions. European Union A...
13,Bangladesh,3,205,preamble,"We, the people of Bangladesh, having proclaime..."
16,Belgium,5,11,preamble,"In the name of the Belgian people, the Nationa..."
19,Bhutan,3,112,preamble,"WE, the people of Bhutan: BLESSED by the Tripl..."
20,Bolivia,0,338,preamble,"In ancient times mountains arose, rivers moved..."
23,Brazil,0,77,preamble,We the representatives of the Brazilian People...
30,Canada,4,20145,preamble,CONSTITUTION ACT An Act for the Union of Canad...


### 5. Listing of Countries Classified as 'Unclear'

We list every country whose document type was classified as `unclear` along with its word count, whether it matched structural keywords or preamble cues, and the first 150 characters of its text.

In [12]:
df_unclear = df_audit[df_audit['doc_type'] == 'unclear'][
    ['Country', 'cluster', 'words', 'starts_with_structure', 'has_preamble_cue', 'first_150_chars']
].copy()

def reason_unclear(r):
    if r['starts_with_structure'] and r['has_preamble_cue']:
        return "Both True (Structure + Cue)"
    elif not r['starts_with_structure'] and not r['has_preamble_cue']:
        return "Both False (Neither Structure nor Cue)"
    return "Other"

df_unclear['Reason'] = df_unclear.apply(reason_unclear, axis=1)

print(f"Total countries with doc_type == 'unclear': {len(df_unclear)}")
display(df_unclear[['Country', 'cluster', 'words', 'Reason', 'first_150_chars']])

Total countries with doc_type == 'unclear': 33


,Country,cluster,words,Reason,first_150_chars
7,Armenia,1,78,Both False (Neither Structure nor Cue),"The Armenian People, accepting as a basis the ..."
10,Azerbaijan,3,186,Both False (Neither Structure nor Cue),"The Azerbaijan people, continuing the traditio..."
22,Botswana,4,30727,Both True (Structure + Cue),CHAPTER I The Republic ss - . Declaration of R...
31,Cabo Verde,4,610,Both False (Neither Structure nor Cue),The Proclamation of National Independence is o...
32,Central African Rep.,6,427,Both False (Neither Structure nor Cue),The Central African People Proud of their nati...
35,China,3,1160,Both False (Neither Structure nor Cue),China is one of the countries with the longest...
42,Croatia,5,642,Both True (Structure + Cue),I Historical Foundations I Historical Foundati...
43,Cuba,0,457,Both False (Neither Structure nor Cue),"WE, CUBAN CITIZENS, heirs and continuators of ..."
55,Estonia,1,57,Both False (Neither Structure nor Cue),"The Esthonian people, with unshaken faith and ..."
69,Guinea-Bissau,0,344,Both False (Neither Structure nor Cue),"In an exemplary manner, the PAIGC, having been..."
